In [1]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt
import struct
import copy

In [2]:
# from open3d import *
import open3d as o3d

In [3]:
from utils_3d import *

from detection_v2_1 import *
from attack_utils import *

### Kitti Coordinates Convention
<img src="./kitti_convention.png" width=400 height=400 />

In [4]:
def alter_obj_pos(obj_pcd, deltaX = 0, deltaY = 0, deltaZ = 0, angle = 0, draw = False) :
    # no change to intensity
    obj_points = np.asarray(obj_pcd.points)
    new_obj_points = []
    new_obj_points = np.reshape(new_obj_points,(-1,3))
    c = np.cos(angle)
    s = np.sin(angle)
    O_x = np.amin(obj_points,axis=0)[0]
    O_y = np.amax(obj_points,axis=0)[1]
    print(O_x, O_y)
    t = [[1,0,0,-O_x],[0,1,0,-O_y],[0,0,1,0],[0, 0, 0,1]]
    t_inv = [[1,0,0,O_x],[0,1,0,O_y],[0,0,1,0],[0, 0, 0,1]]
    r = [[c, -s, 0,1], [s, c, 0,1], [0, 0, 1,0],[0, 0, 0,1]]
    transform = np.array(np.dot(np.dot(t_inv,r),t))
    for point in obj_points :
        points = [point[0], point[1], point[2] , 1]
        points = np.dot(transform,points)
        new_point = np.zeros(3)
        new_point[0] = points[0] + deltaX
        new_point[1] = points[1] + deltaY
        new_point[2] = points[2] + deltaZ
        new_point = np.reshape(new_point, (-1,3))
        new_obj_points = np.append(new_obj_points, new_point, axis = 0)       
    new_obj_pcd = o3d.geometry.PointCloud()
    new_obj_pcd.points = o3d.utility.Vector3dVector(new_obj_points)
    if draw :
        o3d.visualization.draw_geometries([obj_pcd , new_obj_pcd])
    return new_obj_pcd

def get_linesets_point_rays(obj_pcd):
    obj_points = np.array(obj_pcd.points)
    end_points = []
    points = [[0,0,0]]
    for point in obj_points :
        grad = (point[2]/point[0])
        line = Line(grad)
        x = line.get_x(-1.73)
        end_points.append([x, point[1], -1.73])
        points.append([x, point[1], -1.73])
    lines_ = [[0, i] for i in range(1,len(obj_points))]
    line_set = o3d.geometry.LineSet(points=o3d.utility.Vector3dVector(points), 
                                            lines=o3d.utility.Vector2iVector(lines_))
    return line_set , end_points

class Cell(object) :
    def __init__ (self, center_, width=0.5) :
        self.center = center_ # x,y,z
        self.corners = [
            [self.center[0]-width, self.center[1]-width, self.center[2]-width], # bottom left corner
            [self.center[0]+width, self.center[1]-width, self.center[2]-width], # bottom right corner
            [self.center[0]+width, self.center[1]+width, self.center[2]-width], # top right corner
            [self.center[0]-width, self.center[1]+width, self.center[2]-width], # top left corner
            [self.center[0]-width, self.center[1]-width, self.center[2]+width], # bottom left corner
            [self.center[0]+width, self.center[1]-width, self.center[2]+width], # bottom right corner
            [self.center[0]+width, self.center[1]+width, self.center[2]+width], # top right corner
            [self.center[0]-width, self.center[1]+width, self.center[2]+width], # top left corner
        ]
        self.occupied = False
        bbox_ = o3d.geometry.AxisAlignedBoundingBox()
        self.bbox = bbox_.create_from_points(o3d.utility.Vector3dVector(self.corners))
    
    def set_free(self):
        self.occupied = False
        return
    def set_occupied(self):
        self.occupied = True
        return
    def is_occupied(self) :
        return self.occupied
    def print_corners(self) :
        print(self.corners)
    def point_in_cell(self, pt) :
        # generate faces
        f1 =  Face([Vector(self.corners[0]), Vector(self.corners[1]), Vector(self.corners[5]), Vector(self.corners[4])])
        f2 =  Face([Vector(self.corners[3]), Vector(self.corners[7]), Vector(self.corners[6]), Vector(self.corners[2])])
        f3 =  Face([Vector(self.corners[0]), Vector(self.corners[4]), Vector(self.corners[7]), Vector(self.corners[3])])
        f4 =  Face([Vector(self.corners[1]), Vector(self.corners[2]), Vector(self.corners[6]), Vector(self.corners[5])])
        f5 =  Face([Vector(self.corners[4]), Vector(self.corners[5]), Vector(self.corners[6]), Vector(self.corners[7])])
        f6 =  Face([Vector(self.corners[0]), Vector(self.corners[3]), Vector(self.corners[2]), Vector(self.corners[1])])
        
        poly = [f1, f2 ,f3 ,f4 ,f5, f6]
        if isInPoly(pt, poly) :
            return True
        else:
            return False
    def occupancy_check(self, points):
        for pt in points:
            if self.point_in_cell(pt) :
                self.occupied = True
                break
        return self.occupied
    def occupancy_check2(self, pcd) :
        points = np.asarray(pcd.points)
        res = self.bbox.get_point_indices_within_bounding_box(o3d.utility.Vector3dVector(points))
        return res

def angle_between_vecs(vec1, vec2):
    #https://math.stackexchange.com/questions/974178/how-to-calculate-the-angle-between-2-vectors-in-3d-space-given-a-preset-function
    angle = np.arccos((vec1.dot(vec2))/((vec1.norm())*(vec2.norm())))
    return angle

def angle_between_pts(pt1, pt2) :
    angle = np.arctan2([pt1[0]-pt2[0]], [pt1[1]-pt2[1]])
    return angle

import random
def prune_obj_points(obj_pcd, obj_intensity, start_x, start_y, budget=200, skip=False) :
    if skip :
        return obj_pcd, obj_intensity
    
    else:
        pcd_pts = copy.deepcopy(np.asarray(obj_pcd.points)).tolist()
        pcd_intensity = copy.deepcopy(obj_intensity)
        remove_idx = []
        candidate_idx = []
        
        # candidate points (within 10 deg)
        for i in range(0, len(pcd_pts)) :
            pt = pcd_pts[i]
       
            # remove if point does not obey azimuth constraint
            rad = np.sqrt(pt[0]**2 + start_y**2)
            delta_y = np.sqrt(rad**2 + rad**2 -(2*(rad**2)*np.cos(np.radians(10))))

            if pt[1] >= start_y-delta_y :
                #print(pt[1], start_y-delta_y)
                candidate_idx.append(i)
                continue
        
        # randomly remove points from bbox
        if len(pcd_pts) <= budget :
            # if pcd has less than budget pts, all pts are perturbed
            budget = len(pcd_pts)
        
        if len(candidate_idx) <= budget :
            # if candidate_idx has less than budget pts, all pts are perturbed
            budget = len(candidate_idx)
            
        remove = random.sample(candidate_idx, budget)
        remove_idx += remove

        unique = list(set(remove_idx))
        removed_pts = []
        removed_intensity = []
        for index in sorted(unique, reverse=True):
            removed_pts.append(pcd_pts[index])
            removed_intensity.append(pcd_intensity[index])
            pcd_pts.pop(index)
            pcd_intensity =  np.delete(pcd_intensity, index)
            
        if len(pcd_pts) == 0:
            print('All pts removed!')
            
        pcd_intensity = pcd_intensity.tolist()
        # inject removed points behind bbox
        for pt, intensity in zip(removed_pts, removed_intensity) :
            vec = Vector(pt)
            vec_length = vec.norm()
            # get unit vector
            unit_vec = Vector(vec.unit_vec())
            # increment vec by random distance
            new_vec = unit_vec.multi((vec_length+np.random.uniform(2,3)))
            # get new point coordinates
            new_pt = new_vec.array
            # add new point coords to PCD
            pcd_pts.append(new_pt)
            pcd_intensity.append(intensity)
                            
                        
        new_obj_pcd = o3d.geometry.PointCloud()
        new_obj_pcd.points = o3d.utility.Vector3dVector(np.asarray(pcd_pts))
        return new_obj_pcd, pcd_intensity, removed_pts


In [8]:
### this code segment is to modify(translate and rotate) the object inject point cloud

label_folder = "data/datasets/KITTI/training_labels/label_2/"
calib_folder = "data/datasets/KITTI/data_object_calib/training/calib/"
lidar_folder = "data/datasets/KITTI/data_object_velodyne/training/velodyne/"


objs = [('car', 'Car', 125), ('ped','Ped', 70), ('cyl','Cyc', 2612)]
pt_attack = ['10', '20', '40', '60', '100', '150', '200']
for obj in objs :
    ## pcd index to add object to
    num = str(obj[2])
    num = num.zfill(6)
    print(num)
    label_file = label_folder + num + ".txt"
    calib_file = calib_folder + num + ".txt"
    lidar_file = lidar_folder + num + ".bin"

    objects, obj_coords, pcd, pcd_intensity = get_obj_data_intensity(label_file, calib_file, lidar_file, show_plots = False )
    ROI, draw1, draw2, intensity_list = get_obj_regions_intensity(objects, obj_coords, pcd, pcd_intensity, show_plots = False)
    
    for pt_budget in pt_attack: 
        print(pt_budget)
        discarded_pt = []
        added_obj_pcd = []
        for i in range(len(objects)) :
            if not objects[i].type[:3] == obj[1] or np.asarray(ROI[i].points).size == 0:
                continue
            else :
                obj_pcd = copy.deepcopy(ROI[i])
                obj_int = copy.deepcopy(intensity_list[i])
                box = o3d.geometry.AxisAlignedBoundingBox()
                bbox = box.create_from_points(o3d.utility.Vector3dVector(obj_coords[i]))
                object_pt_index = bbox.get_point_indices_within_bounding_box(o3d.utility.Vector3dVector(pcd.points))
                #print(np.asarray(object_pt_index).shape)
                discarded_pt += object_pt_index
                min_x = np.amin(np.asarray(obj_pcd.points),axis=0)[0]
                max_y = np.amax(np.asarray(obj_pcd.points),axis=0)[1]
                obj_p, obj_intensity_p, _ = prune_obj_points(obj_pcd, obj_int, min_x, max_y, budget = int(pt_budget), skip = False)
                #print(np.asarray(obj_p.points).shape)
                added_obj_pcd.append((obj_p, obj_intensity_p))

        # remove all cars from PCD
        unique = list(set(discarded_pt))
        print("pts of cars removed :", str(np.asarray(unique).shape))
        print("new pcd added :", str(len(added_obj_pcd)))
        altered_pcd_points = np.asarray(copy.deepcopy(pcd).points).tolist()
        altered_pcd_intensity = copy.deepcopy(pcd_intensity).tolist()

        for index in sorted(unique, reverse=True):
            altered_pcd_points.pop(index)
            altered_pcd_intensity =  np.delete(altered_pcd_intensity, index)

        # add attacked car pcd + shifted points
        altered_pcd_points = np.asarray(altered_pcd_points)
        if type(altered_pcd_intensity) is np.ndarray:
            altered_pcd_intensity = altered_pcd_intensity.tolist()
        for obj_ in added_obj_pcd :
            print("pt added for obj: ", str(np.asarray(obj_[0].points).shape))
            altered_pcd_points = np.append(altered_pcd_points,np.asarray(obj_[0].points),0)
            altered_pcd_intensity += obj_[1]

        altered_pcd = o3d.geometry.PointCloud()
        altered_pcd.points = o3d.utility.Vector3dVector(altered_pcd_points)

        lidar_file = "output/data2/obj_removal_attack/front_near/"+pt_budget+"pt/"+obj[0]+"/"+num+".bin"
        bin_array = convert_to_kitti_bin(altered_pcd, altered_pcd_intensity, save_file_path = lidar_file)

000125
10
pts of cars removed : (2369,)
new pcd added : 6
pt added for obj:  (2139, 3)
pt added for obj:  (50, 3)
pt added for obj:  (13, 3)
pt added for obj:  (43, 3)
pt added for obj:  (111, 3)
pt added for obj:  (13, 3)
File Saved as : output/data2/obj_removal_attack/front_near/10pt/car/000125.bin
20
All pts removed!
All pts removed!
pts of cars removed : (2369,)
new pcd added : 6
pt added for obj:  (2139, 3)
pt added for obj:  (50, 3)
pt added for obj:  (13, 3)
pt added for obj:  (43, 3)
pt added for obj:  (111, 3)
pt added for obj:  (13, 3)
File Saved as : output/data2/obj_removal_attack/front_near/20pt/car/000125.bin
40
All pts removed!
All pts removed!
pts of cars removed : (2369,)
new pcd added : 6
pt added for obj:  (2139, 3)
pt added for obj:  (50, 3)
pt added for obj:  (13, 3)
pt added for obj:  (43, 3)
pt added for obj:  (111, 3)
pt added for obj:  (13, 3)
File Saved as : output/data2/obj_removal_attack/front_near/40pt/car/000125.bin
60
All pts removed!
All pts removed!
All